# Caso 1 · Cauce: clasificación automática de mensajes de clientes

**Equipo:** _completar nombres_

Este notebook sigue las consignas del enunciado (pasos 1 a 20). Cada paso tiene su celda; agreguen todas las que necesiten.
Las funciones de `src/` se completan en los archivos `.py`, no en el notebook. Antes de entregar: *Reiniciar y ejecutar todo*.

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload vuelve a cargar los módulos de src/ cada vez que los modifican, sin reiniciar el notebook.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42

from src.limpieza import limpiar_texto
from src.naive_bayes import NaiveBayesMultinomial, terminos_mas_indicativos
from src.derivacion import REVISION, derivar, evaluar_derivacion

### Estado de las pruebas automáticas
Ejecuten esta celda cada vez que terminen una función de `src/`.

In [ ]:
!python -m pytest -q --tb=no -rf

## Parte A · Exploración y línea base

**Paso 1.** Carguen `datos/mensajes_historicos.csv` e informen filas, nulos, mensajes vacíos, duplicados (mismo `id_cliente` y mismo `texto`), distribución de `equipo_final`, distribución por canal y cantidad de palabras por clase.

In [ ]:
# Carga del dataset histórico
df = pd.read_csv("datos/mensajes_historicos.csv")

# 1. Total de filas y columnas
print(f"Total de filas: {df.shape[0]:,}")
print(f"Total de columnas: {df.shape[1]}")
print(f"Columnas disponibles: {list(df.columns)}")

# 2. Valores nulos por columna
print("\n--- Valores Nulos por Columna ---")
print(df.isnull().sum())

# 3. Mensajes vacíos o de solo espacios en blanco
mensajes_vacios = (df["texto"].astype(str).str.strip() == "").sum()
print(f"\nMensajes vacíos o de solo espacios en blanco: {mensajes_vacios}")

# 4. Duplicados (mismo id_cliente y mismo texto)
duplicados = df.duplicated(subset=["id_cliente", "texto"]).sum()
print(f"Mensajes duplicados (mismo id_cliente y mismo texto): {duplicados}")

# 5. Distribución de equipo_final (clase objetivo)
print("\n--- Distribución de equipo_final ---")
dist_equipo = pd.DataFrame({
    "Cantidad": df["equipo_final"].value_counts(),
    "Porcentaje (%)": (df["equipo_final"].value_counts(normalize=True) * 100).round(2)
})
display(dist_equipo)

# 6. Distribución por canal de contacto
print("\n--- Distribución por canal ---")
dist_canal = pd.DataFrame({
    "Cantidad": df["canal"].value_counts(),
    "Porcentaje (%)": (df["canal"].value_counts(normalize=True) * 100).round(2)
})
display(dist_canal)

# 7. Cantidad de palabras por clase (equipo_final)
df["cant_palabras"] = df["texto"].astype(str).str.split().str.len()
stats_palabras = df.groupby("equipo_final")["cant_palabras"].agg(
    Media="mean",
    Mediana="median",
    Desvio="std",
    Min="min",
    Max="max"
).round(2)
print("\n--- Cantidad de palabras por clase ---")
display(stats_palabras)


**Paso 2.** Línea base de la mesa de entrada (`equipo_derivado` contra `equipo_final`): exactitud, F1 macro, matriz de confusión normalizada por fila y porcentaje de fraude derivado a otro equipo.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Métricas globales de la mesa de entrada
acc_base = accuracy_score(df["equipo_final"], df["equipo_derivado"])
f1_base = f1_score(df["equipo_final"], df["equipo_derivado"], average="macro")

print("=== LÍNEA BASE: MESA DE ENTRADA HUMANA ===")
print(f"Exactitud (Accuracy): {acc_base:.4f} ({acc_base * 100:.2f}%)")
print(f"F1 Macro:             {f1_base:.4f} ({f1_base * 100:.2f}%)\n")

# Matriz de confusión normalizada por fila (proporción sobre el total real de cada clase)
clases = sorted(df["equipo_final"].unique())
cm_norm = confusion_matrix(df["equipo_final"], df["equipo_derivado"], labels=clases, normalize="true")
df_cm = pd.DataFrame(
    cm_norm,
    index=[f"Real: {c}" for c in clases],
    columns=[f"Derivado: {c}" for c in clases]
).round(4)

print("Matriz de confusión normalizada por fila (tasa de acierto y desvíos):")
display(df_cm)

# Análisis de fraude derivado a otro equipo
total_fraude = (df["equipo_final"] == "fraude").sum()
fraude_mal_derivado = ((df["equipo_final"] == "fraude") & (df["equipo_derivado"] != "fraude")).sum()
pct_fraude_fuga = (fraude_mal_derivado / total_fraude) * 100

print(f"Total casos de Fraude:                {total_fraude}")
print(f"Fraude derivado a OTRO equipo:        {fraude_mal_derivado}")
print(f"Porcentaje de Fraude derivado a otro: {pct_fraude_fuga:.2f}%")


**Paso 3.** Eliminen los duplicados conservando el primero y expliquen por qué se hace antes de dividir.

> **Justificación metodológica:** La deduplicación por `(id_cliente, texto)` debe realizarse **estrictamente antes de la partición de datos (train/test split)** para prevenir el fenómeno de **fuga de datos (data leakage)**. Si no se removieran previamente, un mensaje idéntico emitido por un mismo cliente podría quedar asignado simultáneamente al conjunto de entrenamiento y al de prueba. Esto provocaría una evaluación sesgada sobre datos ya memorizados por el modelo, sobreestimando artificialmente las métricas de exactitud y F1 macro sin reflejar su desempeño real ante nuevos mensajes.

In [ ]:
# Eliminación de duplicados conservando la primera ocurrencia
filas_antes = len(df)
df = df.drop_duplicates(subset=["id_cliente", "texto"], keep="first").copy()
filas_despues = len(df)
eliminados = filas_antes - filas_despues

print(f"Filas iniciales:                 {filas_antes:,}")
print(f"Duplicados eliminados:           {eliminados:,}")
print(f"Filas finales (sin duplicados):  {filas_despues:,}")


## Parte B · Limpieza de texto (`src/limpieza.py`)

**Paso 4.** Implementen `quitar_tildes` y `limpiar_texto` en `src/limpieza.py` hasta que pasen `tests/test_limpieza.py`. Muestren aquí 5 ejemplos de texto original y limpio.

In [ ]:
# Ejemplos representativos aplicando las reglas de normalización y limpieza
ejemplos = [
    "¡Hola! ¿Cómo están? NO PUEDO ENTRAR a mi cuenta...",
    "Me cobraron $ 15.000 y transferí $ 8.350,50 ayer dos veces",
    "Pagué Gs. 150.000 de luz y son USD 20 de comisión",
    "Mi DNI es 30.123.456, operación N° 5543219 hace 3 días",
    "Entré a https://cauce-premios.xyz/login y escribí a soporte@cauce.com.ar 😡👍"
]

tabla_ejemplos = pd.DataFrame({
    "Texto Original": ejemplos,
    "Texto Limpio (limpiar_texto)": [limpiar_texto(ej) for ej in ejemplos]
})

display(tabla_ejemplos)


**Paso 5.** Creen la columna `texto_limpio`. Los mensajes con menos de dos palabras se excluyen del entrenamiento y van a revisión humana: ¿cuántos son y qué contienen?

> **Análisis:** Se identificaron **70 mensajes** (0.88% del conjunto limpio) con menos de dos palabras útiles. Contienen cadenas vacías o de sólo espacios, signos de puntuación aislados (`?`, `...`), emojis solitarios que son suprimidos en la normalización o palabras sueltas sin carga contextual suficiente. Excluirlos del entrenamiento evita inducir ruido en las probabilidades a priori y condicionales de Naive Bayes, canalizándolos apropiadamente a revisión humana.

In [ ]:
# Crear columna texto_limpio
df["texto_limpio"] = df["texto"].apply(limpiar_texto)

# Conteo de palabras sobre el texto limpio
df["palabras_limpias"] = df["texto_limpio"].str.split().str.len()

# Identificar mensajes con menos de dos palabras
mask_cortos = df["palabras_limpias"] < 2
df_cortos = df[mask_cortos]

print(f"Total de mensajes sin duplicados: {len(df):,}")
print(f"Mensajes con menos de 2 palabras limpias: {len(df_cortos):,} ({len(df_cortos)/len(df)*100:.2f}%)")

print("\nDistribución por clase de los mensajes cortos:")
display(df_cortos["equipo_final"].value_counts())

print("\nMuestra de mensajes con menos de 2 palabras (enviados a revisión humana):")
display(df_cortos[["id_mensaje", "canal", "texto", "texto_limpio", "equipo_final"]].head(10))


## Parte C · Naive Bayes desde cero (`src/naive_bayes.py`)

**Paso 6.** Implementen `log_suma_exp`, `NaiveBayesMultinomial` y `terminos_mas_indicativos` hasta que pasen `tests/test_naive_bayes.py`.

In [ ]:
# Tu código aquí

**Paso 7.** División 80/20 estratificada por `equipo_final` con `random_state=42`.

In [ ]:
# Tu código aquí

**Paso 8.** `CountVectorizer` + su clase y la de scikit-learn con `alpha=1`: muestren que predicen lo mismo en el 100% de los mensajes de prueba.

In [ ]:
# Tu código aquí

**Paso 9.** Los 10 términos más indicativos de cada clase y su interpretación para el negocio.

In [ ]:
# Tu código aquí

## Parte D · Ajuste y comparación con KNN

**Paso 10.** `GridSearchCV` del pipeline `CountVectorizer` + `NaiveBayesMultinomial` (5 particiones estratificadas, `f1_macro`). Conclusión sobre cada hiperparámetro.

In [ ]:
# Tu código aquí

**Paso 11.** Misma búsqueda con `TfidfVectorizer` + `KNeighborsClassifier(metric="cosine", algorithm="brute")`. ¿Por qué coseno y no euclídea?

In [ ]:
# Tu código aquí

**Paso 12.** Tiempo de predicción de 1.000 mensajes con 25%, 50%, 75% y 100% del entrenamiento, para los dos modelos. Gráfico y explicación.

In [ ]:
# Tu código aquí

## Parte E · Regla de derivación y evaluación (`src/derivacion.py`)

**Paso 13.** Implementen `derivar` y `evaluar_derivacion` hasta que pasen `tests/test_derivacion.py`.

In [ ]:
# Tu código aquí

**Paso 14.** Probabilidades fuera de muestra con `cross_val_predict` y elección de los dos umbrales (con margen de seguridad) para Naive Bayes y para KNN.

In [ ]:
# Tu código aquí

**Paso 15.** Evaluación única en prueba: tabla con los cinco indicadores de aceptación para Naive Bayes, KNN y la mesa de entrada; reporte de clasificación y matriz de confusión del modelo elegido.

In [ ]:
# Tu código aquí

**Paso 16.** Análisis de 10 errores de derivación automática: causa de cada uno y una mejora concreta.

In [ ]:
# Tu código aquí

**Paso 17.** Impacto en un día de 6.500 mensajes.

In [ ]:
# Tu código aquí

## Parte F · Predicción de los mensajes nuevos

**Paso 18.** Reentrenen el modelo elegido con todos los mensajes históricos limpios y apliquen la regla a `datos/mensajes_nuevos.csv`.

In [ ]:
# Tu código aquí

**Paso 19.** Generen `predicciones_caso1.csv` (2.040 filas; columnas `id_mensaje`, `equipo_predicho`, `prob_max`, `prob_fraude`, `derivacion`) y verifiquen el formato.

In [ ]:
# Tu código aquí

## Parte G · Informe ejecutivo

**Paso 20.** El informe para Laura Benítez se entrega aparte, en `informe_ejecutivo.pdf`.

## Preguntas de reflexión profesional

Respondan cada pregunta en un párrafo, apoyándose en sus resultados.

**1. Naive Bayes supone que las palabras son independientes dada la clase. ¿Por qué clasifica bien igual? ¿Qué consecuencia tiene esa suposición sobre las probabilidades?**

_Respuesta:_ 

**2. ¿Qué pasaría con `alpha=0` si aparece una palabra que nunca se vio en la clase `fraude`?**

_Respuesta:_ 

**3. ¿Por qué `equipo_derivado` no puede usarse como variable del modelo?**

_Respuesta:_ 

**4. ¿Qué costo tendría KNN en producción comparado con Naive Bayes? ¿En qué situación elegirían KNN igual?**

_Respuesta:_ 

**5. ¿Quién debería fijar el máximo de fraude perdido y la exactitud mínima? ¿Qué información tiene que llevar el equipo de datos?**

_Respuesta:_ 

**6. Si Cauce lanza un seguro para celulares, ¿qué le pasa al modelo y cómo lo detectarían en producción?**

_Respuesta:_ 

**7. ¿Qué datos personales anonimizarían antes de guardar los datos de entrenamiento y cómo afectaría al modelo?**

_Respuesta:_ 